# ML-KEM-768 basemul CPA (비밀키 ŝ 전체 복구)

수집한 전력 트레이스 `.npz`를 불러와, decapsulation의 첫 secret-key basemul
(`polyvec_basemul_acc_montgomery`, ŝ[0]·NTT(u[0]))에 CPA를 수행한다.

- 타깃 중간값: `montgomery_reduce(a1 · b1)` (PQClean `basemul`의 첫 `fqmul`)
  - `a1 = ŝ[VEC][2P+1]` : 비밀키 계수 (미지, 복구 대상), NTT 도메인, 0..q-1
  - `b1 = NTT(u)[VEC][2P+1]` : 암호문에서 나오는 알려진 값
- 누출 모델: 16비트 int16 결과의 Hamming weight
- 산출물: 정답 계수의 순위(PGE), 상관 vs 추측값, 상관 트레이스, 트레이스 수별 수렴

수집은 `mlkem-trace-basemul.ipynb`, 복호 포팅 검증은 `mlkem-test.ipynb` 참고.

## 1. PQClean clean 산술 포팅 (기기 중간값과 비트 단위 일치)

zetas·Montgomery·NTT를 `ml-kem-768/clean` 소스 그대로 옮긴 것. `b̂ = poly_ntt(Decompress(c1))`와 정답 `ŝ = poly_frombytes(dk)`가 kyber-py와 mod q로 일치함을 확인했다 (mlkem-test.ipynb).

In [1]:
import numpy as np                              # 수치 계산

Q, QINV, N, K = 3329, -3327, 256, 3               # q=모듈러스, QINV=q^-1 mod 2^16, N=계수수, K=다항식수(768)
ZETAS = [                                         # NTT twiddle 상수 128개 (ntt.c와 동일해야 b̂ 재현됨)
    -1044, -758, -359, -1517, 1493, 1422, 287, 202, -171, 622, 1577, 182, 962, -1202,
    -1474, 1468, 573, -1325, 264, 383, -829, 1458, -1602, -130, -681, 1017, 732, 608,
    -1542, 411, -205, -1571, 1223, 652, -552, 1015, -1293, 1491, -282, -1544, 516, -8,
    -320, -666, -1618, -1162, 126, 1469, -853, -90, -271, 830, 107, -1421, -247, -951,
    -398, 961, -1508, -725, 448, -1065, 677, -1275, -1103, 430, 555, 843, -1251, 871,
    1550, 105, 422, 587, 177, -235, -291, -460, 1574, 1653, -246, 778, 1159, -147, -777,
    1483, -602, 1119, -1590, 644, -872, 349, 418, 329, -156, -75, 817, 1097, 603, 610,
    1322, -1285, -1465, 384, -1215, -136, 1218, -1335, -874, 220, -1187, -1659, -1185,
    -1530, -1278, 794, -1510, -854, -870, 478, -108, -308, 996, 991, 958, -1460, 1522, 1628,
]

def to_i16(x):                                      # 값을 int16 범위로 wrap (보드가 int16_t라 똑같이 잘라야 함)
    return ((int(x) + 0x8000) & 0xFFFF) - 0x8000  # +2^15 후 2^16 마스크 후 -2^15 = 부호있는 16비트

def montgomery_reduce(a):                         # a * R^-1 mod q (R=2^16) — reduce.c와 비트 단위 동일
    t = to_i16(to_i16(a) * QINV)                      # t = (int16_t)((int16_t)a * QINV)
    return (int(a) - t * Q) >> 16                 # (a - t*q) >> 16 (산술 시프트)

def fqmul(a, b):                                  # NTT 도메인 곱 = mont(a*b)
    return montgomery_reduce(to_i16(a) * to_i16(b))   # 두 int16을 곱해 몽고메리 리덕션

def barrett_reduce(a):                            # poly_reduce: ntt 뒤 보드가 적용하는 reduce (대표값 맞춤)
    v = ((1 << 26) + Q // 2) // Q                 # barrett 상수 round(2^26 / q)
    t = (v * to_i16(a) + (1 << 25)) >> 26           # t = round(a / q)
    return to_i16(to_i16(a) - to_i16(t * Q))            # a - t*q (centered 대표값)

def ntt(r):                                       # 정방향 NTT (normal -> bitreversed order)
    r = [to_i16(x) for x in r]                      # 입력을 int16로 (보드와 동일 표현)
    k, length = 1, 128                            # k=zeta 인덱스, length=butterfly 거리
    while length >= 2:                            # 길이 128,64,...,2 로 7단
        start = 0                                 # 각 단의 그룹 시작 위치
        while start < 256:                        # 256계수를 그룹 단위로 훑음
            zeta = ZETAS[k]; k += 1               # 이 그룹의 twiddle
            for j in range(start, start + length):# 그룹 내 butterfly
                t = fqmul(zeta, r[j + length])    # 짝 계수에 twiddle 곱
                r[j + length] = to_i16(r[j] - t)    # 아래쪽 = 위 - t
                r[j] = to_i16(r[j] + t)             # 위쪽 = 위 + t
            start = j + length + 1                # 다음 그룹 시작 (start + 2*length)
        length >>= 1                              # 다음 단은 거리 절반
    return r                                       # NTT 결과 리스트 반환

def poly_frombytes(a):                            # dk의 12비트 패킹을 NTT 비밀키 계수로 복원 (0..q-1)
    r = [0] * N                                   # 256계수 버퍼
    for i in range(N // 2):                       # 3바이트가 2계수
        r[2*i]   = ((a[3*i] >> 0) | (a[3*i+1] << 8)) & 0xFFF   # 하위 12비트 = 계수 0
        r[2*i+1] = ((a[3*i+1] >> 4) | (a[3*i+2] << 4)) & 0xFFF # 상위 12비트 = 계수 1
    return r                                       # ŝ 한 다항식

def polyvec_decompress_vec(a):                    # c1 압축(du=10) 해제: 320바이트 -> 256 normal 계수
    r = [0] * N; p = 0                            # r=출력, p=바이트 포인터
    for j in range(N // 4):                       # 5바이트가 4계수(10비트×4)
        t = ((a[p] | a[p+1] << 8),                # 10비트 값 4개 추출
             (a[p+1] >> 2 | a[p+2] << 6),
             (a[p+2] >> 4 | a[p+3] << 4),
             (a[p+3] >> 6 | a[p+4] << 2))
        p += 5                                    # 다음 5바이트로
        for kk in range(4):                       # 4계수 각각 Decompress
            r[4*j+kk] = ((t[kk] & 0x3FF) * Q + 512) >> 10   # round(q/2^10 * t)
    return r                                       # u 한 다항식 (normal 도메인)

def bhat_from_ct(ct):                             # 아는 값 b̂ = poly_ntt(Decompress(c1)) — 보드 poly_ntt와 동일
    out = []                                      # 다항식 K개
    for i in range(K):                            # 각 u 벡터 성분
        coeffs = polyvec_decompress_vec(ct[320*i:320*(i+1)])  # c1의 i번째 320바이트 -> u_i
        coeffs = ntt(coeffs)                      # 정방향 NTT (반환값 사용)
        out.append([barrett_reduce(x) for x in coeffs])       # ntt 뒤 poly_reduce 적용
    return out                                     # b̂ (NTT 도메인)

def shat_from_dk(dk):                             # 정답 비밀키 ŝ (채점용) — dk 앞 384*K 바이트
    return [poly_frombytes(dk[384*i:384*(i+1)]) for i in range(K)]  # 다항식 K개

print("port loaded")                              # 로드 확인

port loaded


## 2. basemul 트레이스 `.npz` 불러오기

`mlkem-trace-basemul.ipynb`가 저장한 `*-basemul-*.npz`를 우선 선택한다. 새 명명 규칙 이전 파일은 `capture_target`이 없는 legacy A 캡처로만 fallback한다.

In [ ]:
import pathlib                                  # trace 디렉터리 경로와 파일명 검사
import zipfile                                  # 저장 도중인 불완전 npz 파일 제외

trace_directory = pathlib.Path("../traces")     # 수집 노트북이 저장하는 폴더
basemul_candidates = sorted(                    # 새 규칙 basemul 수집 파일만 검색
    candidate for candidate in trace_directory.glob("mlkem*-basemul-*.npz")
    if "tvla" not in candidate.name            # fixed/random TVLA 파일 제외
    and "recovered_shat" not in candidate.name # 분석 결과 파일 제외
    and zipfile.is_zipfile(candidate)           # zip 저장 완료 파일만 사용
)
if not basemul_candidates:                      # A 수집 파일이 아직 없을 때만 legacy 검색
    legacy_candidates = sorted(                 # 타깃 메타가 없는 구형 A 수집 파일 후보
        candidate for candidate in trace_directory.glob("mlkem*-*.npz")
        if "tvla" not in candidate.name        # TVLA 파일은 `waves` 구조이므로 제외
        and "recovered_shat" not in candidate.name # CPA 출력 npz 제외
        and "-poly_tomsg-" not in candidate.name # B 수집 파일 제외
        and "-basemul-" not in candidate.name   # 새 A 파일 패턴은 legacy가 아님
        and zipfile.is_zipfile(candidate)        # 완성된 npz인지 확인
    )
    if not legacy_candidates:                    # 읽을 수 있는 구형 캡처도 없으면
        raise FileNotFoundError("basemul 캡처 .npz가 없음") # 잘못된 파일을 임의로 고르지 않음
    basemul_candidates = legacy_candidates       # 구형 A 캡처만 fallback 허용
    print("legacy basemul 파일 fallback")       # 메타 없는 예전 파일임을 명시

path = basemul_candidates[-1]                   # 이름순 최신 캡처 선택
print("loading:", path)                          # 선택 파일 경로 표시
PARAMS, IMPL, OPT = 768, "clean", "3"           # metadata 이전 캡처용 기본값

with np.load(path) as data:                      # 로드 후 파일 핸들을 닫아 Windows 잠금 방지
    required_fields = {"ct", "dk"}              # 공격 입력과 순위 채점 키 필수
    missing_fields = required_fields.difference(data.files) # 실제 없는 필드만 계산
    if missing_fields:                           # 캡처 파일이 아닌 경우를 분명히 보고
        raise KeyError(f"basemul 캡처 필드 누락: {sorted(missing_fields)}; npz keys={data.files}")
    capture_target = str(data["capture_target"].item()) if "capture_target" in data.files else "legacy-basemul" # target scalar
    if capture_target not in ("basemul", "legacy-basemul"): # 새 메타가 있으면 A 타깃만 허용
        raise ValueError(f"capture_target이 basemul이 아님: {capture_target}") # B를 CPA에 넣지 않음
    if "traces" in data.files:                   # 일반 trace 캡처 배열명
        traces = data["traces"].astype(np.float32) # (파형, 샘플) 형식으로 통일
    elif "waves" in data.files:                  # TVLA npz는 waves로 저장되어 있음
        if "labels" not in data.files:           # TVLA 그룹 구분 라벨이 없으면
            raise KeyError("TVLA npz에 labels 필드가 없음") # trace/waves 파일 오선택 진단
        labels = data["labels"]                  # 각 파형의 fixed/random class
        fixed_count = int(np.count_nonzero(labels == 1)) # fixed class 수 계산
        random_count = int(np.count_nonzero(labels == 0)) # random class 수 계산
        raise ValueError(                             # CPA는 통상 chosen-ct별 평문이 달라야 하므로
            f"선택된 파일은 basemul TVLA 데이터(waves/labels, "
            f"fixed={fixed_count}, random={random_count})입니다. "
            "CPA에는 mlkem-trace-basemul.ipynb가 만든 traces/ct/dk 파일을 선택하세요."
        )
    else:                                          # 파형배열이 아예 없을 경우
        raise KeyError(f"basemul 캡처에 traces 또는 waves 없음; npz keys={data.files}")
    cts = data["ct"]                              # 각 파형과 정렬된 암호문
    dk = data["dk"].tobytes()                    # 채점용 비밀키 bytes
    if "impl" in data.files:                     # 새 수집 메타가 포함된 파일이면
        PARAMS = int(data["scheme"])              # ML-KEM 파라미터셋
        IMPL = str(data["impl"].item())           # 구현 이름
        OPT = str(data["opt"].item())             # 컴파일 최적화 수준
        saved_samples = int(data["samples"])      # 수집 설정상 샘플 수
        saved_adc_mul = int(data["adc_mul"])      # ADC 배수
        print("meta -> target:", capture_target, " scheme:", PARAMS, " impl:", IMPL,
              " opt: -O" + OPT, " samples:", saved_samples, " adc_mul:", saved_adc_mul)

if traces.ndim != 2 or cts.ndim != 2 or len(traces) != len(cts): # 정렬·차원 확인
    raise ValueError(f"traces/ct 형식 또는 행 수 불일치: {traces.shape}, {cts.shape}") # CPA 실행 차단
T, S = traces.shape                             # T=파형 수, S=샘플 수
print("traces:", traces.shape, " ct:", cts.shape) # 실제 분석 입력 shape 표시


## 3. 공격 타깃 설정

`VEC`번째 비밀키 다항식, 계수쌍 `P`의 홀수 계수 `a1 = ŝ[VEC][2P+1]`을 복구한다. A 타깃(첫 basemul)은 `VEC=0`. 알려진 값 `b1`은 파형마다 암호문에서 계산한다.

In [ ]:
VEC, P = 0, 7                                     # 타깃 좌표: a1 = ŝ[VEC][2P+1] (0<=P<128, A타깃은 VEC=0)

true_a1 = shat_from_dk(dk)[VEC][2*P + 1]          # 채점용 정답 계수 (rank 매길 때만 사용)
# b1[t] = b̂[VEC][2P+1] : 파형 t의 "아는 값"을 암호문에서 보드와 동일하게 재현
b1 = np.array([bhat_from_ct(bytes(ct))[VEC][2*P + 1] for ct in cts], dtype=np.int64)  # 파형마다 1개
print("true a1 =", true_a1, " | b1[:5] =", b1[:5])  # 정답과 b1 앞부분 확인

## 4. CPA

추측 `g=0..q-1`마다 모델 `HW16(montgomery_reduce(g·b1))`을 만들고, 각 샘플과 Pearson 상관 →
샘플축 최대 |상관|을 그 추측의 점수로 삼는다. 점수 1등이 복구값.

**주의 — a0/a1 ghost:** basemul은 `b1`을 두 번 곱한다.
`r[0]=fqmul(a1,b1)` (우리 타깃)과 `r[1]=fqmul(a0,b1)`. 모델 `HW(mont(g·b1))`은 둘 다에 반응해서,
전체 구간이면 더 세게 새는 쪽(보통 짝수 계수 a0)이 ghost로 1등을 뺏는다. 두 연산은 **다른 샘플**에서
새므로, 셀 5 상관 트레이스에서 a1 피크 위치를 보고 `WINDOW`를 그 구간으로 좁히면 a1이 1등이 된다.

In [ ]:
WINDOW = (1900, 1940)                                     # None=전체 샘플, (lo,hi)=그 구간만 (a0 ghost 분리용). 예: (3790, 3860)

HW16_TABLE = np.array([bin(v).count("1") for v in range(1 << 16)], dtype=np.uint8)  # 0..65535의 비트1 개수 표

def montgomery_reduce_vec(x):                     # 벡터화 montgomery_reduce (reduce.c와 비트 단위 동일)
    x  = x.astype(np.int64)                        # x = guess*b1 (오버플로 방지 int64)
    lo = x.astype(np.int16)                        # (int16_t)x
    t  = (lo.astype(np.int32) * QINV).astype(np.int16)        # t = (int16_t)(lo*QINV)
    return ((x - t.astype(np.int64) * Q) >> 16).astype(np.int16)  # (x - t*q)>>16

key_guesses = np.arange(Q)                        # 추측 0..3328 (전수)
guess_x_b1  = key_guesses[:, None].astype(np.int64) * b1[None, :]  # (추측 × 트레이스) 모든 곱
hw_model    = HW16_TABLE[montgomery_reduce_vec(guess_x_b1).astype(np.uint16)].astype(np.float64)  # 예측 HW
model_centered = hw_model - hw_model.mean(axis=1, keepdims=True)   # 추측별 평균 제거(상관 분자용)
model_norm     = np.sqrt((model_centered ** 2).sum(axis=1)) + 1e-12  # 추측별 norm (+eps: guess=0 상수 방지)

lo, hi = (0, S) if WINDOW is None else WINDOW     # 분석할 샘플 구간
win = traces[:, lo:hi].astype(np.float64)         # 그 구간 파형만 슬라이스
win_centered = win - win.mean(axis=0)             # 샘플별 평균 제거
win_norm     = np.sqrt((win_centered ** 2).sum(axis=0)) + 1e-12   # 샘플별 norm
corr = (model_centered @ win_centered) / (model_norm[:, None] * win_norm[None, :])  # (추측 × 샘플) Pearson

abs_corr              = np.abs(corr)              # 부호 무시(+/- 누출 둘 다)
best_corr_per_guess   = abs_corr.max(axis=1)      # 추측별 점수 = 샘플축 최대 |상관|
peak_sample_per_guess = abs_corr.argmax(axis=1) + lo  # 그 최대가 난 샘플(전체 파형 기준 보정)

ranked_guesses = np.argsort(best_corr_per_guess)[::-1]           # 점수 높은 순 추측 정렬
true_rank      = int(np.where(ranked_guesses == true_a1)[0][0])  # 정답 등수 (0=1등=복구)
recovered_a1   = int(ranked_guesses[0])           # CPA가 고른 키(1등)
print(f"window = {(lo, hi)}")                     # 분석 구간
print(f"recovered a1 = {recovered_a1}   true a1 = {true_a1}   rank(0=best) = {true_rank}")  # 결과
print("top5 (guess, score, peak_sample):")        # 상위 5개 진단(ghost 위치 확인용)
for g in ranked_guesses[:5]:                      # 점수 상위 5 추측
    tag = "  <= TRUE" if g == true_a1 else ""     # 정답 표시
    print(f"  g={int(g):4d}  score={best_corr_per_guess[g]:.4f}  @sample {int(peak_sample_per_guess[g])}{tag}")

## 5. 그래프: 상관 vs 추측값, 상관 트레이스

상관 트레이스는 **전체 구간**으로 그린다 — a1 피크와 a0 ghost 피크가 다른 샘플에 보이므로, 여기서 a1 피크 위치를 읽어 셀 4의 `WINDOW`를 정한다.

In [ ]:
from bokeh.plotting import figure, show                      # 인터랙티브 플롯
from bokeh.io import output_notebook                          # 노트북 출력
from bokeh.resources import INLINE                             # JS 인라인
from bokeh.layouts import gridplot                             # 두 플롯 가로 배치
from bokeh.models import BoxAnnotation, HoverTool              # 윈도우 음영, 호버

output_notebook(INLINE)                                         # 출력 모드 켜기

full_centered = traces.astype(np.float64) - traces.astype(np.float64).mean(axis=0)  # 전체 파형 평균 제거
full_norm     = np.sqrt((full_centered ** 2).sum(axis=0)) + 1e-12   # 샘플별 norm
corr_true = np.abs((model_centered[true_a1]      @ full_centered) / (model_norm[true_a1]      * full_norm))  # 정답 추측 상관트레이스
corr_best = np.abs((model_centered[recovered_a1] @ full_centered) / (model_norm[recovered_a1] * full_norm))  # 1등 추측 상관트레이스

# 왼쪽: 추측별 CPA 점수 (상관 vs 추측값)
p1 = figure(width=760, height=380, title="CPA score per guess",
            x_axis_label="key guess", y_axis_label="max |corr|")
p1.line(key_guesses, best_corr_per_guess, line_color="gray", line_width=1, legend_label="all guesses")
p1.scatter([true_a1], [best_corr_per_guess[true_a1]], size=9, color="red", legend_label=f"true a1={true_a1}")
p1.add_tools(HoverTool(tooltips=[("guess", "$x{0}"), ("|corr|", "$y{0.0000}")]))
p1.legend.click_policy = "hide"                                 # 범례 클릭 = 숨기기/보이기

# 오른쪽: 상관 트레이스 (전체 구간) — a1/a0 ghost 피크 위치 비교
x = np.arange(len(corr_true))
p2 = figure(width=760, height=380, title="correlation trace (full)",
            x_axis_label="sample", y_axis_label="|corr|")
p2.line(x, corr_true, line_color="navy",   line_width=1, legend_label=f"true a1={true_a1}")
p2.line(x, corr_best, line_color="orange", line_width=1, alpha=0.7, legend_label=f"best g={recovered_a1}")
if WINDOW is not None:                                          # 윈도우 지정 시 음영 박스
    p2.add_layout(BoxAnnotation(left=WINDOW[0], right=WINDOW[1], fill_color="orange", fill_alpha=0.15))
p2.add_tools(HoverTool(tooltips=[("sample", "$x{0}"), ("|corr|", "$y{0.0000}")]))
p2.legend.click_policy = "hide"

show(gridplot([[p1, p2]]))                                      # 가로 배치로 표시

## 6. 트레이스 수별 수렴 (정답 계수의 순위)

셀 4의 `WINDOW` 안에서, 트레이스를 늘리며 정답 계수의 rank가 0으로 떨어지는지 본다. A측 CPA feasibility의 정량 근거.

In [ ]:
from bokeh.plotting import figure, show                      # 인터랙티브 플롯
from bokeh.io import output_notebook                          # 노트북 출력
from bokeh.resources import INLINE                             # JS 인라인
from bokeh.models import Span, HoverTool                       # 기준선, 호버

output_notebook(INLINE)                                         # 출력 모드 켜기

steps = [i for i in range(2, 21)]  # 늘려볼 트레이스 수
rank_curve = []                                   # 각 n의 rank 기록

for n in steps:                                   # 트레이스 n개만으로 반복
    model_n          = hw_model[:, :n]            # 앞 n개 트레이스의 모델
    model_n_centered = model_n - model_n.mean(axis=1, keepdims=True)       # 평균 제거
    model_n_norm     = np.sqrt((model_n_centered ** 2).sum(axis=1)) + 1e-12  # norm

    win_n          = traces[:n, lo:hi].astype(np.float64)                  # 앞 n개 × 셀4 WINDOW 구간
    win_n_centered = win_n - win_n.mean(axis=0)                            # 평균 제거
    win_n_norm     = np.sqrt((win_n_centered ** 2).sum(axis=0)) + 1e-12    # norm

    corr_n  = (model_n_centered @ win_n_centered) / (model_n_norm[:, None] * win_n_norm[None, :])  # 상관
    score_n = np.abs(corr_n).max(axis=1)          # 추측별 점수
    rank_n  = int((score_n > score_n[true_a1]).sum())  # 정답보다 높은 추측 수(0=복구)
    rank_curve.append(rank_n)                     # 기록
    print(f"N={n:6d}  rank={rank_n}")             # 진행 표시

p = figure(width=800, height=400, title=f"CPA convergence (VEC={VEC}, P={P}, window={(lo, hi)})",
           x_axis_label="traces", y_axis_label="rank of true coeff (0=recovered)")
p.line(steps, rank_curve, line_color="navy", line_width=1.5)
p.scatter(steps, rank_curve, size=5, color="navy")
p.add_layout(Span(location=0, dimension="width", line_color="red", line_dash="dashed"))  # rank 0 기준선
p.add_tools(HoverTool(tooltips=[("traces", "$x{0}"), ("rank", "$y{0}")]))
show(p)

## 7. 전체 비밀키 복구 (모든 VEC·P, a0/a1)

셀 3~4는 `ŝ[0][15]` 한 계수만 복구했다. 여기서는 `K=3`개 다항식 × `N/2=128`개 쌍의
`a0=ŝ[VEC][2P]`, `a1=ŝ[VEC][2P+1]` 전부(총 768계수)를 같은 방식으로 반복한다.

**방법:** 각 계수마다
1. (feasibility 전용, 윈도우 보정) 정답 모델 `HW16(mont(true·b))`로 전체 샘플 상관을 구해 피크 샘플을 찾고, 그 주변 `±half_width`를 윈도우로 삼는다 — 셀 4에서 사람이 그래프 보고 `WINDOW=(3800,3850)`를 고른 것을 자동화한 것.
2. 그 윈도우 안에서 `q=3329`개 추측 전수 CPA(셀 4와 동일 로직)로 블라인드 복구.

**주의 — 윈도우 보정은 ŝ를 참조한다:** 실제 블라인드 공격이라면 피크 위치를 모를 때 전체 샘플에서 top-2 피크(= a0/a1 ghost 쌍)를 찾아 시간순서로 구분해야 한다. 여기서는 feasibility 입증(복구 자체가 가능한지)이 목적이라 위치 찾기만 ŝ로 보정하고, 값 복구는 여전히 3329개 전수 블라인드로 한다 (셀 3~4와 같은 수준의 "알려진 정답으로 채점"일 뿐 키를 직접 쓰지는 않음).

In [ ]:
import time

# ── 트레이스 개수 조절 (원본 오염 없이) ──────────────────────────────────
# traces/cts를 직접 덮으면 재실행마다 줄어들어 NT를 다시 못 키움.
# → 원본을 traces_full/cts_full에 '최초 1회만' 보관하고, 항상 거기서 새로 슬라이스.
if "traces_full" not in globals():        # 최초 1회만 원본 백업 (재실행해도 보존)
    traces_full, cts_full = traces, cts
NT = 1000                                  # ★ 쓸 트레이스 수
NT = min(NT, len(traces_full))             # 원본보다 크게 넣어도 안전 clamp
traces = traces_full[:NT]                  # 항상 '원본'에서 새로 슬라이스 (오염 없음)
cts    = cts_full[:NT]
print(f"using {NT} / {len(traces_full)} traces")

# 전체 파형 평균 제거(부호 유지 — abs 아님). 피크 탐색/부호 캘리브/랭킹에 공통 사용
full_centered = traces.astype(np.float64) - traces.astype(np.float64).mean(axis=0)   # (T, S)
full_norm     = np.sqrt((full_centered ** 2).sum(axis=0)) + 1e-12                      # 샘플별 norm

bhats = np.array([bhat_from_ct(bytes(ct)) for ct in cts])   # (T, K, N) 모든 트레이스의 b̂ (한 번만 계산해 재사용)
shat_true = np.array(shat_from_dk(dk))                       # (K, N) 정답 ŝ (피크 캘리브/채점용)
print("bhats:", bhats.shape, " shat_true:", shat_true.shape)

HALF_WIDTH = 3        # 랭킹 윈도우 반폭(샘플). adc_mul=2(2×) 기준 ±3이 최적(검증 753/768). 4×면 ±8. 2×는 lobe가 붙어 ±4만 돼도 Q-c ghost 유입→실패
SCORE_WEAK = 0.2      # |corr| 이 미만이면 "신호 없음(캡처 부족 추정)"으로 표시

def model_for(VEC_, c):                                       # 계수 c의 (모델_중앙화, norm, 정답값) 반환
    # '첫 fqmul = 깨끗한 레지스터 쓰기'인 강한 누출을 겨냥:
    #   홀수 c = a1 -> fqmul(a1, b1),  b1 = b̂[c]      (r[0] 첫 연산, |corr|~0.86)
    #   짝수 c = a0 -> fqmul(a0, b1),  b1 = b̂[c+1]    (r[1] 첫 연산, |corr|~0.94)  ← 약한 a0·b0(0.57) 대신
    b_idx = c if (c % 2 == 1) else c + 1                       # 짝수는 같은 쌍(2P,2P+1)의 홀 인덱스 b̂ 사용
    b  = bhats[:, VEC_, b_idx].astype(np.int64)                # 이 계수가 곱해지는 b1 (트레이스별)
    tv = int(shat_true[VEC_, c])                               # 정답값 (피크 캘리브/채점용; 블라인드 랭킹엔 안 씀)
    gxb  = key_guesses[:, None].astype(np.int64) * b[None, :]  # (Q, T) 모든 추측 × b1
    hw   = HW16_TABLE[montgomery_reduce_vec(gxb).astype(np.uint16)].astype(np.float64)  # HW16(mont(g·b1))
    hw_c = hw - hw.mean(axis=1, keepdims=True)                 # 추측별 평균 제거
    hw_n = np.sqrt((hw_c ** 2).sum(axis=1)) + 1e-12            # 추측별 norm
    return hw_c, hw_n, tv

def corr_true(hw_c, hw_n, tv, lo_s=0, hi_s=None):             # 정답 추측의 '부호 포함' 상관 트레이스
    hi_s = S if hi_s is None else hi_s                        # 탐색 구간 [lo_s, hi_s)
    seg  = full_centered[:, lo_s:hi_s]                        # 구간 파형
    return lo_s, (hw_c[tv] @ seg) / (hw_n[tv] * full_norm[lo_s:hi_s])   # (시작오프셋, 부호 상관)

def cpa_rank(hw_c, hw_n, tv, peak, sign, half=HALF_WIDTH):   # peak '단일 샘플' 부호 상관으로 랭킹
    lo_c, hi_c = max(0, peak - half), min(S, peak + half)     # 창은 norm 계산용으로만
    win   = traces[:, lo_c:hi_c].astype(np.float64)
    win_c = win - win.mean(axis=0)
    win_n = np.sqrt((win_c ** 2).sum(axis=0)) + 1e-12
    corr  = (hw_c @ win_c) / (hw_n[:, None] * win_n[None, :]) # (Q, win) 부호 상관
    pk    = min(peak - lo_c, corr.shape[1] - 1)               # 창 안에서 peak 위치(보통 half; 경계 clamp)
    score = sign * corr[:, pk]                                # ★ max 아니라 '그 peak 샘플 하나'의 부호 상관
    ranked = np.argsort(score)[::-1]                          # Q-c는 반대부호라 음수 → 최하위로
    return int(ranked[0]), int(np.where(ranked == tv)[0][0]), float(np.abs(corr).max())


results = []                                                  # (VEC, c, recovered, true, rank, peak, |corr|)
total = K * N
t0 = time.time()

# ── 누출 부호 캘리브(핵심) ──────────────────────────────────────────────
# HW가 전력과 +/- 어느 쪽으로 상관되는지는 기기 특성(전 계수 공통 상수).
# 누출은 쌍극(+lobe/-lobe)이라 한 계수만 보면 노이즈로 부호가 뒤집힘
#  -> VEC0 앞쪽 센 계수들의 (argmax|corr| 지점 부호)를 |corr|로 가중 '다수결' 해서 안정적으로 결정
sign_vote = 0.0
for c in range(min(20, N)):
    hw_c, hw_n, tv = model_for(0, c)
    _, cr = corr_true(hw_c, hw_n, tv)
    k = int(np.argmax(np.abs(cr)))                            # 이 계수의 가장 센 지점
    sign_vote += np.sign(cr[k]) * abs(cr[k])                  # |corr| 가중 부호 투표
LEAK_SIGN = float(np.sign(sign_vote)) or 1.0                  # +1 또는 -1 (이후 전 계수 공통)
print(f"leak sign calibrated (weighted majority): {LEAK_SIGN:+.0f}")

def find_peak(hw_c, hw_n, tv, lo_s=0, hi_s=None):            # LEAK_SIGN 방향으로 가장 큰 샘플 = 일관된 lobe
    off, cr = corr_true(hw_c, hw_n, tv, lo_s, hi_s)
    k = int(np.argmax(LEAK_SIGN * cr))                        # |corr|가 아니라 '부호 방향' 최대 (lobe 일관성 보장)
    return off + k

# 1) VEC=0: 신호 뚜렷 → 전체 S에서 피크 탐색 + '계수 c ↔ 샘플' 선형관계 캘리브
vec0_peaks = []
for c in range(N):
    hw_c, hw_n, tv = model_for(0, c)
    peak = find_peak(hw_c, hw_n, tv)
    recovered, rank, mag = cpa_rank(hw_c, hw_n, tv, peak, LEAK_SIGN)
    if mag < SCORE_WEAK:                                       # 신호가 노이즈 수준이면
        rank = -1                                             #   -1 = "신호 없음(캡처 부족 추정)"
    results.append((0, c, recovered, tv, rank, peak, mag))
    vec0_peaks.append(peak)
    if len(results) % 64 == 0:
        print(f"{len(results):4d}/{total}  VEC=0 c={c:3d} rec={recovered:4d} true={tv:4d} rank={rank} |corr|={mag:.3f}  ({time.time()-t0:.1f}s)")

slope, intercept = np.polyfit(np.arange(N), vec0_peaks, 1)     # peak ≈ slope·c + intercept  (쌍당 일정 간격)
print(f"VEC=0 calibration: slope={slope:.2f} samples/coeff, intercept={intercept:.1f}")
prev_hi = max(vec0_peaks)

# 2) VEC=1,2: 한 VEC 블록 길이(slope·N)로 '밴드'를 잡고 그 밴드 전체에서 탐색.
#    밴드가 넓어도 각 계수 모델은 자기 b1에만 강하게 반응 → 엉뚱한 계수/블록에 안 걸리고,
#    블록 사이 poly_add 위치를 몰라도 됨(GAP 추정 불필요). 랭킹은 아래 cpa_rank의 좁은(±HALF_WIDTH) 윈도우가 담당.
block_len = slope * N                                         # 한 VEC 블록의 대략 샘플 길이
prev_end  = max(vec0_peaks)                                   # VEC0 마지막 누출 위치 = 다음 밴드 시작 기준
for VEC_ in (1, 2):
    band_lo = max(0, int(prev_end - slope * 3))               # 이전 블록 끝 살짝 앞부터
    band_hi = min(S, int(prev_end + block_len * 1.3))         # 다음 블록 전체 + 여유(블록간 gap 흡수)
    vec_peaks, vec_mags = [], []
    for c in range(N):
        hw_c, hw_n, tv = model_for(VEC_, c)
        peak = find_peak(hw_c, hw_n, tv, band_lo, band_hi)    # 이 VEC 밴드 안에서 (부호방향) 피크 탐색
        recovered, rank, mag = cpa_rank(hw_c, hw_n, tv, peak, LEAK_SIGN)
        if mag < SCORE_WEAK:
            rank = -1
        results.append((VEC_, c, recovered, tv, rank, peak, mag))
        vec_peaks.append(peak); vec_mags.append(mag)
        if len(results) % 64 == 0:
            print(f"{len(results):4d}/{total}  VEC={VEC_} c={c:3d} rec={recovered:4d} true={tv:4d} rank={rank} |corr|={mag:.3f}  ({time.time()-t0:.1f}s)")
    prev_end = max(vec_peaks)                                 # 다음 VEC 밴드 시작 기준 갱신
    print(f"  VEC={VEC_} band=({band_lo},{band_hi})  mean|corr|={np.mean(vec_mags):.3f}")

ranks = np.array([r[4] for r in results])
ok   = ranks == 0
weak = ranks == -1
print()
print(f"전체 {total}계수 중 복구 성공(rank=0): {int(ok.sum())} / {total}  ({100*ok.mean():.2f}%)")
print(f"신호 없음(캡처 부족 추정, |corr|<{SCORE_WEAK}): {int(weak.sum())}개")
if (~ok & ~weak).any():
    fails = [(r[0], r[1], r[2], r[3], r[4], round(r[6], 3)) for r in results if r[4] not in (0, -1)]
    print("신호는 있는데 복구 실패 (VEC, c, recovered, true, rank, |corr|):")
    for fr in fails[:20]:
        print("  ", fr)

### 복구된 ŝ 조립 및 검증

In [ ]:
recovered_shat = np.zeros((K, N), dtype=np.int64)            # 복구된 비밀키 (NTT 도메인, 전체)
weak_mask = np.zeros((K, N), dtype=bool)                      # score < SCORE_WEAK (신호 없음 추정) 좌표
for VEC_, c, recovered, tv, rank, peak, score in results:
    recovered_shat[VEC_, c] = recovered
    weak_mask[VEC_, c] = (rank == -1)

match = (recovered_shat == shat_true)                         # 계수별 일치 여부 (신호 없는 자리는 당연히 대부분 불일치)
print("계수 단위 일치율(전체):", f"{match.mean()*100:.2f}%", f"({int(match.sum())}/{match.size})")
print("신호 있는 계수 중 일치율:", f"{match[~weak_mask].mean()*100:.2f}%", f"({int(match[~weak_mask].sum())}/{int((~weak_mask).sum())})")
print("전체 키 완전 일치:", bool(match.all()))

if match.all():
    print("[OK] s-hat 768계수 전부 CPA로 복구 성공 -- A측(비밀키 basemul) 전체 키 복구 완료")
elif weak_mask.any():
    print(f"[PARTIAL] 신호 없음(캡처 부족 추정) 좌표 {int(weak_mask.sum())}개 -> SAMPLES 늘려서 재캡처 필요")
    bad = np.argwhere(~match & ~weak_mask)
    print("[FAIL] 신호는 있는데 틀린 좌표 (VEC, c):", bad.tolist()[:20])
else:
    bad = np.argwhere(~match)
    print("[FAIL] 불일치 좌표 (VEC, c):", bad.tolist()[:20])

import pathlib
stamp = time.strftime("%Y%m%d-%H%M%S")
out_path = pathlib.Path("../traces") / f"mlkem{PARAMS}-{IMPL}-O{OPT}-cpa-basemul-recovered_shat-{stamp}.npz"
np.savez_compressed(out_path, recovered_shat=recovered_shat, shat_true=shat_true, match=match, weak_mask=weak_mask)
print("saved ->", out_path)

### 시각화 — 계수별 복구 결과 (bokeh)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.models import HoverTool

output_notebook(INLINE)

idxs = np.arange(len(results))                                # 0..767 (VEC*256 + c 순서)
weak = ranks == -1                                             # 신호 없음(캡처 부족 추정)
fail = (~ok) & (~weak)                                         # 신호는 있는데 틀린 경우
p = figure(width=1000, height=350,
           title=f"전체 비밀키 계수별 CPA 복구 결과 (성공 {int(ok.sum())}/{total}, 신호없음 {int(weak.sum())}, 실패 {int(fail.sum())})",
           x_axis_label="coefficient index (VEC*256 + c)", y_axis_label="rank of true value (0=success, -1=신호없음)")
p.scatter(idxs[ok],   ranks[ok],   size=3, color="green",  legend_label="success")
p.scatter(idxs[fail], ranks[fail], size=6, color="red",    legend_label="fail")
p.scatter(idxs[weak], ranks[weak], size=6, color="orange", legend_label="no signal (capture too short)")
p.add_tools(HoverTool(tooltips=[("idx", "$x{0}"), ("rank", "$y{0}")]))
p.legend.click_policy = "hide"
show(p)